# Feature Engineering: ColumnTransformer and Pipeline

### Do You Want to Build a Snowman?

Prepare the inputs for a model that predicts snowman height.

<img src="./assets/olaf.jpeg" width="300" alt="Snowman illustration">

This deliberately tiny, fictional dataset is a **preprocessing exercise**, not evidence that lunch or dinner predicts snowman height. Treat all rows as a supplied training sample; do not report a model generalization score from these 14 examples.

**Business connection:** in a housing dataset, the same steps could impute missing floor areas, scale numerical measurements, and encode property types before predicting price. The workflow transfers; the useful features and preprocessing choices depend on the task.

#### Load Packages

In [1]:
import numpy as np
import pandas as pd

#### Load Data

In [2]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data).fillna(np.nan)
df_train

,temp,lunch,dinner,precipitation,height_snowman_cm
0,-3.0,soup,pizza,yes,100
1,5.0,sandwich,pizza,no,0
2,0.0,soup,noodles,yes,75
3,7.0,burger,NaN,yes,0
4,3.0,sandwich,fishsticks,yes,20
5,-1.0,soup,pizza,yes,25
6,1.0,cereal,NaN,no,0
7,NaN,salad,fishsticks,yes,35
8,-6.0,sandwich,noodles,yes,170
9,3.0,burger,pizza,no,0


## Exercise

1. Separate `df_train` into `X_train` and `y_train`; the target is `height_snowman_cm`.
2. Identify numeric, nominal, and binary inputs, then inspect missing values.
3. Choose and justify an imputation strategy for each incomplete input. Row order has no time meaning here, so interpolation between neighboring rows is not justified.
4. Choose suitable encodings and a scaler. Explain which columns need each transformation and avoid inventing an order for nominal categories.
5. Combine your transformations using `ColumnTransformer` and `Pipeline`. Decide which steps must run in sequence and which apply to different columns.
6. Produce `X_train_fe` with numerical values, no missing values, the same row count and index as `X_train`, and no target column.

Fit on training data only. To transform new rows later, reuse the fitted objects instead of recomputing statistics. Make sure each input feature is selected exactly once.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
])

In [5]:
target = "height_snowman_cm"
X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

In [6]:
transformers = [
    ("temperature", numeric_pipeline, ["temp"]),
    ("categories", categorical_pipeline, ["lunch", "dinner", "precipitation"]),
]

fe_column_transformer = ColumnTransformer(
    transformers=transformers, remainder="drop"
).set_output(transform="pandas")

fe_column_transformer.fit(X_train)
X_train_fe = fe_column_transformer.transform(X_train)

assert X_train_fe.index.equals(y_train.index)
assert not X_train_fe.isna().any().any()
X_train_fe

,temperature__temp,categories__lunch_burger,categories__lunch_cereal,categories__lunch_salad,categories__lunch_sandwich,categories__lunch_soup,categories__dinner_fishsticks,categories__dinner_noodles,categories__dinner_pizza,categories__precipitation_no,categories__precipitation_yes
0,-1.078720,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
1,1.438293,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
2,-0.134840,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,2.067546,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.809040,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
5,-0.449467,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
6,0.179787,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
7,-0.134840,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
8,-2.022600,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.809040,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


### Check Your Transformations on New Inputs

Reuse your fitted objects to transform the two rows below, **without calling `fit` or `fit_transform` again**. Keep the training output columns and the new row indices, and check that no missing values remain. `wrap` is an unseen lunch category: explain whether your encoder raises an error or handles it explicitly. An expected error is a valid observation; do not silently refit on these rows.

In [8]:
# New inputs only: the fitted transformers must not learn from these rows.
X_new = pd.DataFrame(
    {
        "temp": [np.nan, 2.0],
        "lunch": ["soup", "wrap"],
        "dinner": [np.nan, "pizza"],
        "precipitation": ["yes", "no"],
    },
    index=["known", "new_category"],
)
X_new

,temp,lunch,dinner,precipitation
known,NaN,soup,NaN,yes
new_category,2.0,wrap,pizza,no


In [9]:
X_new_fe = fe_column_transformer.transform(X_new)
assert X_new_fe.columns.equals(X_train_fe.columns)
X_new_fe

,temperature__temp,categories__lunch_burger,categories__lunch_cereal,categories__lunch_salad,categories__lunch_sandwich,categories__lunch_soup,categories__dinner_fishsticks,categories__dinner_noodles,categories__dinner_pizza,categories__precipitation_no,categories__precipitation_yes
known,-0.134840,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
new_category,0.494413,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
